# 01 · Corpus and chunking

Trustworthy RAG on Medical Documents: Boussoualim Yassine, Benahcene Mohamed Wassim, Aissaoui Anes Safouane, Djerdali Mohamed Achraf.

This notebook loads the 50 documents and cuts them into the chunks the index is built from, with the code of `src/ingest.py`. It makes no LLM call.

Before running it: install `requirements.txt`, then download the corpus with `python scripts/fetch_corpus.py --from-manifest` (see the README). The BGE-M3 tokenizer is downloaded on first use.

In [1]:
import os
import sys

# The notebooks live in notebooks/; the project code is in src/.
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

## 1. Load the documents

`load_documents` reads each WHO PDF page by page with `pypdf` (a page keeps its number, for the citation) and each PubMed Central article as one text. Pages with almost no text are skipped.

In [2]:
import collections

import ingest

docs = ingest.load_documents()
by_kind = collections.Counter(d["kind"] for d in docs)
sources = collections.Counter(d["kind"] for d in {d["source"]: d for d in docs}.values())
print(f"{len(docs)} records from {len(set(d['source'] for d in docs))} documents")
for kind in by_kind:
    print(f"  {kind}: {sources[kind]} documents, {by_kind[kind]} records")

1530 records from 50 documents
  WHO: 20 documents, 1500 records
  PubMed: 30 documents, 30 records


In [3]:
# Page 14 of the WHO malaria guideline: the evidence page of question A06.
doc = next(d for d in docs if d["source"].startswith("WHO_09") and d["page"] == 14)
print("source:", doc["source"])
print("page:  ", doc["page"])
print(doc["text"][:600], "...")

source: WHO_09_Guidelines_for_the_treatment_of_malaria.pdf
page:   14
Preventing relapse in P. vivax or P. ovale malaria
The G6PD status of patients should be used to guide administration of primaquine 
for preventing relapse. 
Good practice statement
To prevent relapse, treat P. vivax or P. ovale malaria in children and adults (except 
pregnant women, infants aged < 6 months, women breastfeeding infants aged 
< 6 months, women breastfeeding older infants unless they are known not to 
be G6PD deficient, and people with G6PD deficiency) with a 14-day course 
(0.25-0.5 mg/kg bw daily) of primaquine in all transmission settings.
Strong recommendation, high-quality  ...


## 2. Count tokens with the embedding model's tokenizer

A chunk is measured in tokens of BGE-M3's own tokenizer, so a chunk is exactly what the embedding model reads.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
lengths = sorted(len(tokenizer.encode(d["text"], add_special_tokens=False)) for d in docs)
print(f"{sum(lengths):,} tokens in all")
print("median record:", lengths[len(lengths) // 2], "tokens; longest:", lengths[-1])

1,214,696 tokens in all
median record: 589 tokens; longest: 32197


## 3. Cut into chunks

Windows of `CHUNK_TOKENS` = 512 tokens, moving by 512 − 102 tokens, so consecutive windows share `OVERLAP_TOKENS` = 102 tokens (the brief's 20 %). A window never spans two pages.

In [5]:
chunks = ingest.chunk_document(doc, tokenizer)
print(f"{doc['source']} p. {doc['page']}: {len(chunks)} chunk(s)")
for c in chunks:
    print(f"  start_token={c['start_token']:>4}  {c['text'][:90]!r}...")

WHO_09_Guidelines_for_the_treatment_of_malaria.pdf p. 14: 2 chunk(s)
  start_token=   0  'Preventing relapse in P. vivax or P. ovale malaria The G6PD status of patients should be u'...
  start_token= 410  'tolerate oral therapy, complete treatment with 3 days of ACT (add single dose primaquine i'...


In [6]:
all_chunks = ingest.build_chunks(tokenizer)
per_kind = collections.Counter(c["kind"] for c in all_chunks)
print(f"{len(all_chunks)} chunks:", dict(per_kind))
print("example id:", all_chunks[0]["id"])

3410 chunks: {'WHO': 2668, 'PubMed': 742}
example id: WHO_01_Clinical_guidelines_for_the_management_of_hypertension.pdf|3|0


The id `source|page|start_token` is stable, so an interrupted index build (`src/index.py`) resumes where it stopped. The report (Section 3) explains why all 3,410 chunks are kept although the brief names a maximum of 2,000.